## Table 1. Descriptive Statistics (n = 457)

**Notation:**
M±SD — mean ± standard deviation;
Md [Q1; Q3] — median and interquartile range;
Min–Max — minimum and maximum value;
n (%) — count and proportion for categorical variables.


In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

DATA_PATH        = '../Models/data/features.parquet'
DPI              = 300
FIG_W            = 14
ROW_H, HEADER_H  = 0.28, 0.40
FONTSIZE         = 8.5

CATEGORICAL_COLS = ['sex', 'neuro_deficit', 'facet_tropism']


In [ ]:
LABELS = {
    'sex':                  'Sex',
    'age':                  'Age, years',
    'bmi':                  'BMI, kg/m²',
    'disease_duration':     'Disease duration, mo',
    'neuro_deficit':        'Neurological deficit',
    'back_pain_pre':        'Back pain (VAS)',
    'leg_pain_pre':         'Leg pain (VAS)',
    'sf36_phys_pre':        'SF-36 PCS',
    'sf36_psych_pre':       'SF-36 MCS',
    'sf36_total_pre':       'SF-36 total',
    'odi_pre':              'ODI, %',
    'disc_height':          'Disc height, mm',
    'sagittal_angle':       'Sagittal angle, °',
    'lumbar_lordosis':      'Lumbar lordosis, °',
    'linear_shift':         'Linear shift, mm',
    'pfirrmann':            'Pfirrmann grade',
    'fujiwara':             'Fujiwara grade',
    'adc':                  'ADC, ×10⁻³ mm²/s',
    'facet_angle':          'Facet angle, °',
    'facet_tropism':        'Facet tropism',
    'total_pain_pre':       'Total pain (VAS)',
    'leg_pain_proportion':  'Leg pain ratio',
    'back_pain_proportion': 'Back pain ratio',
    'onset_age':            'Age of onset, years',
    'degeneration_index':   'Degeneration index',
    'shift_height_ratio':   'Shift/height ratio',
    'adc_per_pfirrmann':    'ADC/Pfirrmann',
}


In [ ]:
def build_categorical_df(df):
    rows = []
    for col in CATEGORICAL_COLS:
        counts = df[col].value_counts()
        total  = counts.sum()
        for i, v in enumerate(counts.index):
            rows.append({
                'Variable': LABELS.get(col, col) if i == 0 else '',
                'Value':    v,
                'n':        counts[v],
                '%':        f'{counts[v] / total * 100:.1f}',
            })
    return pd.DataFrame(rows)


def build_numerical_df(df):
    rows = []
    for col in df.columns:
        if col in CATEGORICAL_COLS:
            continue
        s        = df[col].dropna()
        d        = 2 if s.abs().max() < 10 else 1
        mean, sd = s.mean(), s.std(ddof=1)
        med      = s.median()
        q1, q3   = s.quantile(0.25), s.quantile(0.75)
        mn, mx   = s.min(), s.max()
        rows.append({
            'Variable':    LABELS.get(col, col),
            'M±SD':        f'{mean:.{d}f}±{sd:.{d}f}',
            'Md [Q1; Q3]': f'{med:.{d}f} [{q1:.{d}f}; {q3:.{d}f}]',
            'Min–Max':     f'{mn:.{d}f}–{mx:.{d}f}',
        })
    return pd.DataFrame(rows)


def _base_table(ax, table_df, col_widths):
    fig_h = ax.get_figure().get_size_inches()[1]
    tbl   = ax.table(
        cellText=table_df.values.tolist(),
        colLabels=list(table_df.columns),
        cellLoc='center',
        loc='center',
        colWidths=col_widths,
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(FONTSIZE)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_facecolor('white')
        cell.set_edgecolor('black')
        if row == 0:
            cell.set_linewidth(0.8)
            cell.set_text_props(fontweight='bold', fontsize=FONTSIZE)
            cell.set_height(HEADER_H / fig_h)
        else:
            cell.set_linewidth(0.4)
            cell.set_height(ROW_H / fig_h)
            if col == 0:
                cell.set_text_props(ha='left')
                cell.PAD = 0.02
    return tbl


def save_numerical_table(num_df, out_path):
    fig_h = HEADER_H + len(num_df) * ROW_H + 0.2
    fig, ax = plt.subplots(figsize=(FIG_W, fig_h), dpi=DPI)
    ax.axis('off')
    _base_table(ax, num_df, col_widths=[0.36, 0.22, 0.28, 0.14])
    plt.tight_layout(pad=0.3)
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    print(f'Saved: {out_path}')
    plt.close(fig)


def save_categorical_table(cat_df, group_starts, out_path):
    fig_h = HEADER_H + len(cat_df) * ROW_H + 0.2
    fig, ax = plt.subplots(figsize=(FIG_W, fig_h), dpi=DPI)
    ax.axis('off')
    tbl = _base_table(ax, cat_df, col_widths=[0.40, 0.30, 0.15, 0.15])
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            continue
        is_group_start = row in group_starts
        is_last_row    = row == len(cat_df)
        if col == 0:
            cell.visible_edges = 'LR'
            if is_group_start:
                cell.visible_edges += 'T'
            if is_last_row:
                cell.visible_edges += 'B'
        else:
            cell.visible_edges = 'LRTB'
    plt.tight_layout(pad=0.3)
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    print(f'Saved: {out_path}')
    plt.close(fig)


In [ ]:
df = pd.read_parquet(DATA_PATH)

### Categorical variables

In [ ]:
cat_df = build_categorical_df(df)

group_starts = {0}
offset = 1
for col in CATEGORICAL_COLS:
    group_starts.add(offset)
    offset += df[col].nunique()

save_categorical_table(cat_df, group_starts, 'descriptive_statistics_categorical.tiff')


### Numerical variables

In [ ]:
num_df = build_numerical_df(df)
save_numerical_table(num_df, 'descriptive_statistics_numerical.tiff')
